In [1]:
# ── STEP 0: install dependencies (run once) ─────────────────────────────
!pip install crispritz azimuth

# ── STEP 1: imports & load your file ────────────────────────────────────
import ast
import pandas as pd

# point this at your real path
MASTER_CSV = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Embedding/all_embeddings.csv"

df = pd.read_csv(MASTER_CSV)
# expect: df["gRNA"] and df["off_target_summary_data"] exist
# off_target_summary_data is a string like "{0:1,1:1,2:3,3:19,4:194}"

# ── STEP 2: QUICK “approximate” specificity from the counts ─────────────
# here we weight each i-mm off-target by a penalty w_i
penalty = {1:0.1, 2:0.01, 3:0.001, 4:0.0001}

def approx_specificity(s):
    d = ast.literal_eval(s)
    tot = sum(d.get(i,0) * penalty[i] for i in penalty)
    return 1.0 / (1.0 + tot)

df["spec_approx"] = df["off_target_summary_data"].apply(approx_specificity)

# ── (OPTIONAL) STEP 3: TRUE CFD specificity via CRISPRitz ───────────────
from crispritz import Crispritz

# — configure these for your system —
BOWTIE2_INDEX = "/path/to/bowtie2/index/hg38"  # your hg38 (or other) bowtie2 index prefix
PAM           = "NGG"
MAX_MISMATCH  = 4

# run once for all guides
guides = df["gRNA"].astype(str).tolist()
with Crispritz(
    genome_index = BOWTIE2_INDEX,
    pam          = PAM,
    max_mismatch = MAX_MISMATCH,
    compute_cfd  = True
) as cz:
    cz.run_guides(guides)
    ot_df = cz.get_off_targets_dataframe()

# ot_df has columns ["guide_seq","off_target_seq","mm_count","cfd_score",…]
# sum the CFD penalties per guide
cfd_sum = ot_df.groupby("guide_seq")["cfd_score"].sum().rename("cfd_sum")
df = df.join(cfd_sum, on="gRNA")
df["spec_cfd"] = 1.0 / (1.0 + df["cfd_sum"])

# ── (OPTIONAL) STEP 4: ON-TARGET efficiency via Azimuth ─────────────────
from azimuth.model_comparison import predict
# (this will download a small model file first time)
eff = predict.get_efficiency_scores(df["gRNA"].tolist())
df["on_target_efficiency"] = eff

# ── STEP 5: write out your new master ──────────────────────────────────
OUT = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Embedding/all_with_scores.csv"
df.to_csv(OUT, index=False)
print("Done — wrote", OUT)


ERROR: Could not find a version that satisfies the requirement crispritz (from versions: none)
ERROR: No matching distribution found for crispritz


/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_93104/4023391446.py:11: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(MASTER_CSV)


ValueError: malformed node or string: nan